In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.removeAll()

In [0]:
## PARAMETRIZAR CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

In [0]:
def read_tablas_Silver():

    df_tablaproducto = spark.table(f"{PRM_catalogo}.silver.Tabla_Producto") \
        .select(
            col("Cod_producto").alias("Cod_producto"),  
            col("Categoria").alias("Categoria")
        ) 

    df_TablaEcommerce = spark.table(f"{PRM_catalogo}.silver.Tabla_IntEcommerce") \
        .select(
            trim(col("ID_interaccion")).alias("ID_interaccion"),
            date_format(to_timestamp(col("Fecha_Interaccion"), "M/d/yyyy H:mm"), "MM-yyyy").alias("Periodo_Mes"),
            trim(col("Cod_producto")).alias("Cod_producto"),
            trim(col("Cod_Tipo_Interaccion")).alias("Cod_Tipo_Interaccion"),
            col("Puntuacion_Producto").alias("Puntuacion_Producto")         
        ) \
        .where(
                (col("Cod_producto").isNotNull()) 
              )
              
    df_TipInteraccion = spark.table(f"{PRM_catalogo}.silver.Tabla_Destipinteraccion") \
        .select(
            trim(col("Cod_Tipo_Interaccion")).alias("Cod_Tipo_Interaccion"),  
            trim(col("Nombre_codigo_sistema")).alias("Nombre_codigo_sistema"), 
        )


    return df_tablaproducto, df_TablaEcommerce, df_TipInteraccion

In [0]:
def Tablajoin(df_tablaproducto, df_TablaEcommerce, df_TipInteraccion):

    df_join = \
        df_TablaEcommerce.alias("A")\
        .join(df_TipInteraccion.alias("B"), col("A.Cod_Tipo_Interaccion") == col("B.Cod_Tipo_Interaccion"), "left")\
        .join(df_tablaproducto.alias("P"), col("A.Cod_producto") == col("P.Cod_producto"), "left")\
            .select(
                col("A.ID_interaccion").alias("ID_interaccion"),                
                col("A.Cod_producto").alias("Cod_producto"),
                date_format(to_date(col("A.Periodo_Mes"), "MM-yyyy"), "MMMM-yyyy").alias("Periodo_Mes"),              
                col("P.Categoria").alias("Categoria"),
                col("B.Nombre_codigo_sistema").alias("Nombre_CodInteraccion"),
            ).where(col("A.ID_interaccion").isNotNull() & col("A.Cod_producto").isNotNull())

    df_prefinal = df_join\
        .groupBy(
            col("Categoria"),
            col("Nombre_CodInteraccion"),
            col("Periodo_Mes")
        ).agg(
            count("Categoria").alias("CantInteraccion_Categoria")
        )

    df_finalKPI = df_prefinal.withColumn("KPI_Producto_Ecommerce",
            when(col("CantInteraccion_Categoria") > 40, "HIGH")
            .when(col("CantInteraccion_Categoria") > 20, "MEDIUM")
            .otherwise("LOW")
        ).orderBy(col("Categoria").asc(),col("CantInteraccion_Categoria").desc())


    return  df_finalKPI

In [0]:
def main():

    df_tablaproducto, df_TablaEcommerce, df_TipInteraccion = read_tablas_Silver()

    df_finalKPI = Tablajoin(df_tablaproducto, df_TablaEcommerce, df_TipInteraccion)

    df_finalKPI.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.golden.Categoria_Top_Ecommerce")

main()